In [1]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchinfo import summary
from torch import nn    
from torch.nn import functional as F
from torch.utils.data import random_split
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Dataset

In [3]:
DATA_DIR = "../datasets/waveforms"

In [4]:
class WavefromDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.mixture_files = sorted([f for f in os.listdir(directory) if f.endswith('_mixture.npy')])

    def __len__(self):
        return len(self.mixture_files)

    def __getitem__(self, idx):

        mixture_path = os.path.join(self.directory, self.mixture_files[idx])
        vocal_path = mixture_path.replace('_mixture.npy', '_vocals.npy')
        instrumental_path = mixture_path.replace('_mixture.npy', '_instrumental.npy')

        mixture = np.load(mixture_path)
        vocals = np.load(vocal_path)
        instrumental = np.load(instrumental_path)

        X = torch.from_numpy(mixture).float().unsqueeze(0)
        Y = torch.from_numpy(np.stack([vocals, instrumental], axis=0)).float()
        
        pad = (-X.shape[-1]) % 16
        X = F.pad(X, (0, pad))
        Y = F.pad(Y, (0, pad))
        return X, Y

In [5]:
train_dataset = WavefromDataset(os.path.join(DATA_DIR,"train"))
val_dataset = WavefromDataset(os.path.join(DATA_DIR,"val"))

print(len(train_dataset))
print(len(val_dataset))

3592
895


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=13,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=13,
    shuffle=False
)

Model

In [7]:
class Demucs(nn.Module):
    def __init__(self,inchannels = 1, outchannels = 2, features = [32, 64, 128, 256]):
        super().__init__()

        self.features = features
        self.inchannels = inchannels
        self.outchannels = outchannels

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        in_features = inchannels
        for idx, feature in enumerate(self.features):
            en = self.Encoder(in_features, feature)
            self.encoders.append(en)

            decoder_out = self.features[0] if idx == 0 else in_features

            de = self.Decoder(2*feature, decoder_out, final=(idx == 0))
            self.decoders.append(de)

            in_features = feature

        self.decoders = self.decoders[::-1]

        self.bottleneckLSTM = nn.LSTM(
                input_size=self.features[-1],
                hidden_size=self.features[-1] // 2,
                num_layers=4,
                batch_first=True,
                bidirectional=True,
                dropout=0.2)

        self.final = nn.Conv1d(self.features[0],outchannels,kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def Encoder(self,in_channels,out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels,in_channels,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels,2*in_channels,kernel_size=1),
            nn.GLU(dim=1),
            nn.Conv1d(in_channels,out_channels,kernel_size=8,stride=4,padding=1),
            nn.GroupNorm(8,out_channels),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def Decoder(self, in_channels, out_channels, final=False):

        layers = [
            nn.Conv1d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels, 2 * in_channels, kernel_size=1),
            nn.GLU(dim=1),
            nn.ConvTranspose1d(
                in_channels, out_channels,
                kernel_size=8, stride=4, padding=1
            ),
            nn.GroupNorm(8,out_channels)
        ]

        if not final:
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))

        return nn.Sequential(*layers)

    def forward(self,mixture):

        x = mixture
        skip = []

        for en in self.encoders:
            x = en(x)
            skip.append(x)

        x = x.transpose(1, 2)
        x, _ = self.bottleneckLSTM(x)
        x = x.transpose(1, 2)

        skip = skip[::-1]

        for i, de in enumerate(self.decoders):
            diff = skip[i].shape[-1] - x.shape[-1]
            
            if diff > 0:
                x = F.pad(x, (0, diff))
            elif diff < 0:
                x = x[..., :diff]
                
            x = torch.cat((x, skip[i]), dim=1)
            x = de(x)

        diff = mixture.shape[-1] - x.shape[-1]

        if diff > 0:
            x = F.pad(x, (0, diff))
        elif diff < 0:
            x = x[..., :mixture.shape[-1]]

        return self.final(x)

In [8]:
model = Demucs(inchannels=1, outchannels=2).to(device)
summary(model, input_size=(13, 1 , 205008), device=device)

Layer (type:depth-idx)                   Output Shape              Param #
Demucs                                   [13, 2, 205008]           --
├─ModuleList: 1-1                        --                        --
│    └─Sequential: 2-1                   [13, 32, 51251]           --
│    │    └─Conv1d: 3-1                  [13, 1, 205008]           4
│    │    └─ReLU: 3-2                    [13, 1, 205008]           --
│    │    └─Conv1d: 3-3                  [13, 2, 205008]           4
│    │    └─GLU: 3-4                     [13, 1, 205008]           --
│    │    └─Conv1d: 3-5                  [13, 32, 51251]           288
│    │    └─GroupNorm: 3-6               [13, 32, 51251]           64
│    │    └─ReLU: 3-7                    [13, 32, 51251]           --
│    │    └─Dropout: 3-8                 [13, 32, 51251]           --
│    └─Sequential: 2-2                   [13, 64, 12812]           --
│    │    └─Conv1d: 3-9                  [13, 32, 51251]           3,104
│    │    └─R

In [9]:
epochs = 150
patience = 15
best_loss_value = float("inf")
lastbest = -1

loss_fn = nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.75,
    patience=3,
    min_lr=1e-5
)

train_loss_epochwise = []
val_loss_epochwise = []

MODEL_DIR = "../models/"
PLOT_DIR = "../plots/"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

model_name = "demucs_vocal_inst_sep"

model = model.to(device)

In [10]:
def save_loss_plots():
    plt.plot(train_loss_epochwise, color="blue", label="Train Loss")
    plt.plot(val_loss_epochwise, color="orange", label="Val Loss")
    plt.title("Epochwise Train vs Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(os.path.join(PLOT_DIR, f"{model_name}_loss_epochwise.png"))
    plt.close()

In [11]:
def save_checkpoint(path, epoch, model, optimizer, scheduler,
                     train_loss_epochwise, val_loss_epochwise,
                     best_loss_value, lastbest):

    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "train_loss_epochwise": train_loss_epochwise,
        "val_loss_epochwise": val_loss_epochwise,
        "best_loss_value": best_loss_value,
        "lastbest": lastbest,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "numpy_rng_state": np.random.get_state(),
    }

    torch.save(checkpoint, path)
    print(f"Checkpoint saved at epoch {epoch+1} -> {path}")


def load_checkpoint(path, model, optimizer, scheduler, device,
                     start_epoch=0,
                     train_loss_epochwise=None,
                     val_loss_epochwise=None,
                     best_loss_value=float("inf"),
                     lastbest=-1):

    if train_loss_epochwise is None:
        train_loss_epochwise = []
    if val_loss_epochwise is None:
        val_loss_epochwise = []

    if not os.path.exists(path):
        print(f"No checkpoint found, Starting fresh from epoch {start_epoch+1}")
        return {
            "start_epoch": start_epoch,
            "train_loss_epochwise": train_loss_epochwise,
            "val_loss_epochwise": val_loss_epochwise,
            "best_loss_value": best_loss_value,
            "lastbest": lastbest,
        }

    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if checkpoint["cuda_rng_state"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint["cuda_rng_state"]])
    np.random.set_state(checkpoint["numpy_rng_state"])

    print(f"Checkpoint loaded from epoch {checkpoint['epoch']+1} <- {path}")

    return {
        "start_epoch": checkpoint["epoch"],
        "train_loss_epochwise": checkpoint["train_loss_epochwise"],
        "val_loss_epochwise": checkpoint["val_loss_epochwise"],
        "best_loss_value": checkpoint["best_loss_value"],
        "lastbest": checkpoint["lastbest"],
    }

In [12]:
os.makedirs("../checkpoints/", exist_ok=True)
CHECKPOINT_PATH = "../checkpoints/demucs_vocal_inst_sep_ckpt.pt"


state = load_checkpoint(
    CHECKPOINT_PATH, model, optimizer, scheduler, device,
    start_epoch=0,
    train_loss_epochwise=[],
    val_loss_epochwise=[],
    best_loss_value=float("inf"),
    lastbest=-1,
)


start_epoch = state["start_epoch"]
train_loss_epochwise = state["train_loss_epochwise"]
val_loss_epochwise = state["val_loss_epochwise"]
best_loss_value = state["best_loss_value"]
lastbest = state["lastbest"]

No checkpoint found, Starting fresh from epoch 1


In [13]:
for epoch in range(start_epoch, epochs):

    model.train()
    running_train_loss = 0.0

    for batch_idx, (X, Y) in enumerate(train_loader):
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(X)
        loss = loss_fn(prediction, Y)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Step [{batch_idx+1}/{len(train_loader)}] Train Loss: {loss.item():.6f}")

    avg_train_loss = running_train_loss / len(train_loader)
    train_loss_epochwise.append(avg_train_loss)

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for X, Y in val_loader:
            X, Y = X.to(device), Y.to(device)
            prediction = model(X)
            loss = loss_fn(prediction, Y)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_loss_epochwise.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.7f}\n")

    save_loss_plots()
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_loss_value:
        best_loss_value = avg_val_loss
        lastbest = epoch
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_best.pt"))
        print(f"New best model saved: {best_loss_value:.6f}")

    save_checkpoint(CHECKPOINT_PATH, epoch, model, optimizer, scheduler,
                     train_loss_epochwise, val_loss_epochwise,
                     best_loss_value, lastbest)

    if epoch - lastbest >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break


torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_last.pt"))

Epoch [1/150] Step [1/277] Train Loss: 0.496559
Epoch [1/150] Step [51/277] Train Loss: 0.050116
Epoch [1/150] Step [101/277] Train Loss: 0.061052
Epoch [1/150] Step [151/277] Train Loss: 0.046236
Epoch [1/150] Step [201/277] Train Loss: 0.060244
Epoch [1/150] Step [251/277] Train Loss: 0.054736
Epoch [1/150] Train Loss: 0.060397 | Val Loss: 0.040713 | LR: 0.0010000

New best model saved: 0.040713
Checkpoint saved at epoch 1 -> ../checkpoints/demucs_vocal_inst_sep_ckpt.pt
Epoch [2/150] Step [1/277] Train Loss: 0.050231
Epoch [2/150] Step [51/277] Train Loss: 0.061655
Epoch [2/150] Step [101/277] Train Loss: 0.051806
Epoch [2/150] Step [151/277] Train Loss: 0.051504
Epoch [2/150] Step [201/277] Train Loss: 0.050750
Epoch [2/150] Step [251/277] Train Loss: 0.040990
Epoch [2/150] Train Loss: 0.049021 | Val Loss: 0.039439 | LR: 0.0010000

New best model saved: 0.039439
Checkpoint saved at epoch 2 -> ../checkpoints/demucs_vocal_inst_sep_ckpt.pt
Epoch [3/150] Step [1/277] Train Loss: 0.03631